# 01 · Baseline (untrained gemma-3-1b-it)
Answers the 350 test pairs with the **untrained** model, using exactly the prompt and decoding settings of the fine-tuned run.
Output: `eval/baseline_untrained/predictions_test.csv` in Drive. Runtime: GPU (L4 preferred, T4 works), about 5 minutes.

## Setup

In [ ]:
# ---- Settings (edit once) ----
DRIVE_ROOT = "/content/drive/MyDrive/gemma-tr-temporal-inference"
LOCAL = "/content/data"            # local copy of the dataset (Drive I/O is slow)

In [ ]:
# ---- Mount Drive ----
from google.colab import drive, userdata
drive.mount("/content/drive")

In [ ]:
# ---- Pinned libraries (torch comes with Colab and is logged in every *_meta.json) ----
# If a later cell fails with an import error: Runtime > Restart session, then run every cell again except this one.
!pip install -q transformers==4.56.2 trl==0.23.1 peft==0.17.1 accelerate==1.10.1 datasets==4.1.1

In [ ]:
# ---- Hugging Face access ----
# One-time: accept the Gemma licence at https://huggingface.co/google/gemma-3-1b-it
# and add a read token as a Colab secret named HF_TOKEN (key icon on the left).
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
# ---- Copy the frozen dataset (v1) to local disk and check it is unchanged ----
import hashlib, shutil
DATA_FILES = {   # sha256 from data/v1/MANIFEST.json (frozen 2026-10-21)
    "splits/train.csv": "9fa401ea9c8680df38b1255834420c7f60e0f7b5471410fa69e6ef981c563b63",
    "splits/test.csv": "6f374a18735e57741c73a8f1292bc3a441d23142dac82306247cb373d45a327a",
    "structured/structured.csv": "95dc97809eac2722db1c4d2aaea2865a44d9eabc5b229a30c932f06414b60258",
}
for rel, sha in DATA_FILES.items():
    dst = f"{LOCAL}/{rel}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(f"{DRIVE_ROOT}/data/v1/{rel}", dst)
    got = hashlib.sha256(open(dst, "rb").read()).hexdigest()
    assert got == sha, f"STOP: {rel} differs from the frozen v1 file"
print("dataset OK:", ", ".join(DATA_FILES))

In [ ]:
# ---- GPU check ----
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0), "| dtype:", "bf16" if torch.cuda.is_bf16_supported() else "fp32 (T4: slower, still correct)")

## Code

In [ ]:
# ---- Shared settings: identical for the baseline and the fine-tuned run. Do not edit after the first run. ----
import gc, json, os, platform, time
import numpy as np
import pandas as pd

MODEL_ID = "google/gemma-3-1b-it"
SEED = 42
DEV_FRAC = 0.10          # share of TRAIN stories held out for eval loss / checkpoint choice
PROMPT_VERSION = "p1"

# Decoding is greedy and fixed so that baseline and fine-tuned outputs are comparable.
GEN_KWARGS = dict(do_sample=False, max_new_tokens=64, repetition_penalty=1.0)

INSTRUCTION = (
    "Aşağıda bir hikâyenin ortasından alınmış bir bölüm var. "
    "Soru, bu bölümden önce ya da sonra olan bir şeyi soruyor. "
    "Bölümdeki ipuçlarını kullanarak en olası cevabı Türkçe ve tek bir kısa cümleyle yaz."
)


def build_messages(part2, question):
    """Single user turn (Gemma 3 has no system role)."""
    return [{"role": "user", "content": f"{INSTRUCTION}\n\nMetin: {part2}\n\nSoru: {question}"}]


def load_split(data_dir, split):
    df = pd.read_csv(os.path.join(data_dir, "splits", f"{split}.csv"))
    need = {"text_id", "direction", "part2", "question", "answer"}
    missing = need - set(df.columns)
    assert not missing, f"{split}.csv is missing columns {missing}"
    assert df[list(need)].notna().all().all(), f"{split}.csv has empty cells"
    return df


def train_dev_split(train_df, dev_frac=DEV_FRAC, seed=SEED):
    """Hold out whole stories from train (never from test)."""
    ids = np.array(sorted(train_df.text_id.unique()))
    perm = np.random.default_rng(seed).permutation(len(ids))
    n_dev = int(round(dev_frac * len(ids)))
    dev_ids = set(ids[perm[:n_dev]])
    dev = train_df[train_df.text_id.isin(dev_ids)].reset_index(drop=True)
    tr = train_df[~train_df.text_id.isin(dev_ids)].reset_index(drop=True)
    assert not set(tr.text_id) & set(dev.text_id)
    return tr, dev


def to_prompt_completion(df):
    """Conversational prompt-completion format: TRL computes the loss on the completion only."""
    from datasets import Dataset
    return Dataset.from_list([
        {"prompt": build_messages(r.part2, r.question),
         "completion": [{"role": "assistant", "content": r.answer}]}
        for r in df.itertuples()
    ])


def pick_dtype():
    """bf16 on Ampere+ (L4, A100). fp32 on T4: Gemma 3 is unstable in fp16."""
    import torch
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float32


def load_model_and_tokenizer(model_id=MODEL_ID, adapter=None, padding_side="right"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    dtype = pick_dtype()
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = padding_side
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, attn_implementation="eager")
    if adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    return model, tok, dtype


def stop_token_ids(tok):
    ids = {tok.eos_token_id}
    eot = tok.convert_tokens_to_ids("<end_of_turn>")
    if eot is not None and eot != tok.unk_token_id:
        ids.add(eot)
    return sorted(i for i in ids if i is not None)


def env_info():
    import torch, transformers, trl, peft
    return {
        "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "trl": trl.__version__, "peft": peft.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }


def write_json(obj, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def free_gpu():
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ---- Answer a split with the base model, or base model + LoRA adapter ----
def clean(text):
    """Keep the first non-empty line; the task asks for one short sentence."""
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    return lines[0] if lines else ""


def predict(split, system, out, adapter=None, batch_size=16, limit=0, data_dir=LOCAL, model_id=MODEL_ID):
    import torch
    torch.manual_seed(SEED)
    df = load_split(data_dir, split)
    if limit:
        df = df.head(limit)
    model, tok, dtype = load_model_and_tokenizer(model_id, adapter, padding_side="left")
    model.eval()
    stops = stop_token_ids(tok)
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

    raw, t0 = [], time.time()
    for i in range(0, len(df), batch_size):
        chunk = df.iloc[i:i + batch_size]
        texts = [tok.apply_chat_template(build_messages(r.part2, r.question), tokenize=False,
                                         add_generation_prompt=True) for r in chunk.itertuples()]
        # the chat template already contains <bos>; do not add it twice
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        enc = {k: enc[k] for k in ("input_ids", "attention_mask")}
        with torch.no_grad():
            gen = model.generate(**enc, **GEN_KWARGS, eos_token_id=stops, pad_token_id=pad_id)
        raw += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"{min(i + batch_size, len(df))}/{len(df)}", flush=True)

    res = df[["text_id", "direction", "question", "answer"]].copy()
    res["prediction"] = [clean(t) for t in raw]
    res["raw_output"] = raw
    res["system"] = system
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    res.to_csv(out, index=False)
    write_json({
        "system": system, "split": split, "n": len(res), "model_id": model_id, "adapter": adapter,
        "prompt_version": PROMPT_VERSION, "gen_kwargs": GEN_KWARGS, "dtype": str(dtype),
        "seconds": round(time.time() - t0, 1), "empty_predictions": int((res.prediction == "").sum()),
        "env": env_info(),
    }, os.path.splitext(out)[0] + "_meta.json")
    del model
    free_gpu()
    print(f"saved {len(res)} predictions to {out} | empty: {(res.prediction == '').sum()}")
    return res

## Smoke test (8 items)
Check that the answers look like Turkish sentences before the full run.

In [ ]:
predict("test", "baseline", "/content/smoke/pred_test.csv", limit=8)[["question", "answer", "prediction"]]

## Full test set

In [ ]:
OUT = f"{DRIVE_ROOT}/eval/baseline_untrained/predictions_test.csv"
p = predict("test", "baseline", OUT)
p.sample(10, random_state=0)[["question", "answer", "prediction"]]